In [18]:
import os
import h5py
import numpy as np
from astropy.io import fits
from astropy.table import Table
from tqdm.notebook import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

In [19]:
gal_dataset = Table.read('/pscratch/sd/j/jlargett/DESI_SGA_MINE/Sorter/FP/FP_Targets_SSL_wanchors_it2.csv')
cutout_dir = "/pscratch/sd/j/jlargett/DESI_SGA_MINE/Sorter/FP/fpy3_cutouts"
h5_filename = "fpy3_it2.h5"

num_galaxies = 79668
count = num_galaxies
pixel = 152

max_workers = 4      # safer than 8 or 16
chunk_size = 1000    # only submit 1000 jobs at a time
batch_size = 64      # write 64 images at a time

string_dtype = h5py.string_dtype(encoding="utf-8")

In [20]:
def get_path(i):
    return (
        f"{cutout_dir}/"
        f"{int(gal_dataset['targetid'][i])}_grz_152_"
        f"{gal_dataset['target_ra'][i]:.4f}_"
        f"{gal_dataset['target_dec'][i]:.4f}.fits"
    )


def read_one(i):
    path = get_path(i)

    if not os.path.exists(path):
        return None

    with fits.open(path, memmap=False, ignore_missing_simple=True, ignore_missing_end=True) as hdul:
        img_data = hdul[0].data

    if img_data is None or img_data.shape != (3, pixel, pixel):
        return None

    return {
        "image": img_data.astype(np.float32),
        "ra": float(gal_dataset["ra"][i]),
        "dec": float(gal_dataset["dec"][i]),
        "mag_g": float(gal_dataset["mag_g"][i]),
        "mag_r": float(gal_dataset["mag_r"][i]),
        "mag_z": float(gal_dataset["mag_z"][i]),
        "cutout_type": 1,
        "Main_Type": int(gal_dataset["Main_Type"][i]),
        "targetid": int(gal_dataset["targetid"][i]),
    }


with h5py.File(h5_filename, "w") as f:

    images = f.create_dataset(
        "images",
        (count, 3, pixel, pixel),
        dtype="f4",
        chunks=(64, 3, pixel, pixel),
        maxshape=(None, 3, pixel, pixel)
    )

    ra = f.create_dataset("ra", (count,), dtype="f8", maxshape=(None,))
    dec = f.create_dataset("dec", (count,), dtype="f8", maxshape=(None,))
    mag_g = f.create_dataset("mag_g", (count,), dtype="f4", maxshape=(None,))
    mag_r = f.create_dataset("mag_r", (count,), dtype="f4", maxshape=(None,))
    mag_z = f.create_dataset("mag_z", (count,), dtype="f4", maxshape=(None,))

    cutout_type = f.create_dataset("cutout_type", (count,), dtype="i4", maxshape=(None,))
    Main_Type = f.create_dataset("Main_Type", (count,), dtype="i4", maxshape=(None,))
    targetid = f.create_dataset("targetid", (count,), dtype="i8", maxshape=(None,))

    valid_entries = 0
    batch = []

    with tqdm(total=num_galaxies, desc="Saving galaxies") as pbar:

        for start in range(0, len(gal_dataset), chunk_size):

            if valid_entries >= num_galaxies:
                break

            end = min(start + chunk_size, len(gal_dataset))

            with ThreadPoolExecutor(max_workers=max_workers) as executor:
                futures = [executor.submit(read_one, i) for i in range(start, end)]

                for future in as_completed(futures):
                    try:
                        result = future.result()
                    except Exception as e:
                        print(f"Read error: {e}")
                        continue

                    if result is None:
                        continue

                    batch.append(result)

                    if len(batch) >= batch_size:
                        n = len(batch)
                        s = slice(valid_entries, valid_entries + n)

                        images[s] = np.stack([b["image"] for b in batch])
                        ra[s] = [b["ra"] for b in batch]
                        dec[s] = [b["dec"] for b in batch]
                        mag_g[s] = [b["mag_g"] for b in batch]
                        mag_r[s] = [b["mag_r"] for b in batch]
                        mag_z[s] = [b["mag_z"] for b in batch]
                        cutout_type[s] = [b["cutout_type"] for b in batch]
                        Main_Type[s] = [b["Main_Type"] for b in batch]
                        targetid[s] = [b["targetid"] for b in batch]

                        valid_entries += n
                        pbar.update(n)
                        batch = []

            f.flush()

    # Write final leftover batch
    if len(batch) > 0:
        n = len(batch)
        s = slice(valid_entries, valid_entries + n)

        images[s] = np.stack([b["image"] for b in batch])
        ra[s] = [b["ra"] for b in batch]
        dec[s] = [b["dec"] for b in batch]
        mag_g[s] = [b["mag_g"] for b in batch]
        mag_r[s] = [b["mag_r"] for b in batch]
        mag_z[s] = [b["mag_z"] for b in batch]
        cutout_type[s] = [b["cutout_type"] for b in batch]
        Main_Type[s] = [b["Main_Type"] for b in batch]
        targetid[s] = [b["targetid"] for b in batch]

        valid_entries += n

    print(f"Saved {valid_entries} valid galaxies to {h5_filename}.")

Saving galaxies:   0%|          | 0/79668 [00:00<?, ?it/s]

Saved 79668 valid galaxies to fpy3_it2.h5.
